# Resources, Dependencies & Lifecycle

## What's covered

- The **resource** as the atomic unit — what one block represents and what it does not
- **Attribute references** and how Terraform builds a directed acyclic graph from them
- **Implicit vs explicit** dependencies — `depends_on` and when you actually need it
- The **`lifecycle`** meta-arguments — `create_before_destroy`, `prevent_destroy`, `ignore_changes`, `replace_triggered_by`
- **`count`** and **`for_each`** for creating many resources from one block, and why `for_each` almost always wins
- **The `-/+` replace** pattern, revisited — when the plan says "destroy and recreate" and what to do about it
- A worked example — a VPC with three subnets driven by `for_each`


## The resource — Terraform's atomic unit

A `resource` block tells Terraform "manage one piece of infrastructure of this type, with these arguments." The block is named by two strings: the **type** (a name the provider exposes, like `aws_s3_bucket` or `aws_instance`) and the **name** (an identifier you choose, scoped to the module). Together they form the resource's **address**: `aws_s3_bucket.logs`, `aws_instance.web`.

```hcl
resource "aws_s3_bucket" "logs" {
  bucket = "my-app-logs-2025"
}

resource "aws_instance" "web" {
  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.micro"
}
```

The resource address is the unit of every state operation, every plan diff, every `terraform` command. `terraform state list` prints them. `terraform plan` reports changes by address. `terraform import` brings a cloud resource under a specific address. Memorize the shape — it shows up everywhere.

**What a resource is.** A one-to-one mapping to a thing in the cloud — usually a single API object. One `aws_s3_bucket` block creates one bucket. One `aws_instance` block creates one EC2 instance.

**What a resource is not.** A *collection*. If you want three buckets, you need three resource blocks — or one block with `count` or `for_each` (covered later in this notebook).

**Why the type/name split exists.** The type tells the provider what kind of API call to make. The name disambiguates between multiple resources of the same type in the same module. `aws_s3_bucket.logs` and `aws_s3_bucket.uploads` are two different buckets; the provider doesn't care which name you chose, but every reference in your code is by address.


## Attribute references — building the graph

Every resource exposes a set of **attributes** — the values the provider returned when the resource was created. Some are arguments you set (`bucket`, `ami`); others are computed by the cloud (`arn`, `id`, `private_ip`, `public_dns`).

You reference attributes from other resources with **dotted addresses**:

```hcl
resource "aws_s3_bucket" "logs" {
  bucket = "my-app-logs-2025"
}

resource "aws_s3_bucket_policy" "logs" {
  bucket = aws_s3_bucket.logs.id          # reference to the bucket's id attribute
  policy = jsonencode({
    Version = "2012-10-17"
    Statement = [{
      Effect    = "Allow"
      Principal = "*"
      Action    = "s3:GetObject"
      Resource  = "${aws_s3_bucket.logs.arn}/*"   # reference inside a string
    }]
  })
}
```

The reference `aws_s3_bucket.logs.id` says "the `id` attribute of the `aws_s3_bucket` named `logs`." Two consequences fall out of writing that reference:

- **The policy now depends on the bucket.** Terraform must create the bucket before the policy can be created (the bucket's `id` isn't known until then).
- **The policy will be re-evaluated whenever the bucket's `id` changes.** In practice, this means if the bucket is replaced, the policy is updated to point at the new bucket.

The dependency is **implicit** — it exists because of the reference. You did not have to declare it explicitly. This is the dominant case in real Terraform code: 99% of dependencies arise from the references you naturally write to wire resources together.

**Inside strings, use `${...}` interpolation.** Outside strings, use bare references. Both are equivalent — `bucket = aws_s3_bucket.logs.id` and `bucket = "${aws_s3_bucket.logs.id}"` produce the same thing. Modern HCL prefers the bare form when possible.


## The directed acyclic graph

Terraform reads every resource block, finds every attribute reference, and builds a **directed acyclic graph (DAG)** of dependencies. Each node is a resource; each edge is a reference from one resource to another.

```
            +-------------------+
            |  random_id.suffix |
            +---------+---------+
                      |
                      v
              +---------------+
              | aws_s3_bucket |
              |     .hello    |
              +-------+-------+
                      |
                      v
        +-----------------------------+
        | aws_s3_bucket_versioning    |
        |          .hello             |
        +-----------------------------+
```

The graph drives three things:

- **Order.** Terraform processes a resource only after every dependency has finished. The random ID is created first, then the bucket, then versioning.
- **Parallelism.** Resources with no dependencies on each other run in parallel. By default, up to 10 at a time (`-parallelism=10`). Twenty independent IAM users? They get created in batches of 10 concurrently.
- **Replacement propagation.** If a resource gets `-/+` replaced, every resource that references its computed attributes (`id`, `arn`) is re-planned with the new values.

The graph is acyclic by construction — circular references are caught at plan time with a clear error. `terraform graph` renders the DAG as GraphViz dot output if you want to see it:

```bash
$ terraform graph | dot -Tsvg > graph.svg
```

A useful debugging exercise on a tangled codebase. Most "why is this thing being replaced?" questions resolve by looking at the graph.


## `depends_on` — when implicit isn't enough

Sometimes a dependency exists but the code doesn't have a reference to encode it. Two common cases:

**1. A side effect the cloud requires but the API doesn't expose.** Example: an IAM role policy attachment must exist before an EC2 instance that uses the role's instance profile starts. The instance doesn't reference the attachment in HCL — but the cloud will reject the instance launch if the attachment isn't in place yet.

**2. Provisioning order across services.** A Lambda function might need a logging-policy attachment to exist before it's invoked. The Lambda itself doesn't reference the policy.

In both cases, you declare the dependency with the **`depends_on`** meta-argument:

```hcl
resource "aws_iam_role_policy_attachment" "lambda_logs" {
  role       = aws_iam_role.lambda.name
  policy_arn = "arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
}

resource "aws_lambda_function" "worker" {
  function_name = "worker"
  role          = aws_iam_role.lambda.arn
  handler       = "index.handler"
  runtime       = "python3.11"
  filename      = "worker.zip"

  depends_on = [aws_iam_role_policy_attachment.lambda_logs]
}
```

The Lambda's `role` argument references the IAM role, which gives an implicit dependency on the role. But the *policy attachment* on that role isn't referenced anywhere in the Lambda block — so Terraform might create the Lambda before the attachment is in place. The `depends_on` forces the ordering.

**The honest rule.** `depends_on` is a code smell *most of the time*. Reach for it only when the implicit graph is genuinely insufficient — IAM, Lambda permissions, and a few cross-region patterns. The fact that it exists means you should always look for a reference first. If you find yourself writing `depends_on` because "it doesn't work without it," investigate why before committing.

`depends_on` takes a list of resource addresses. It can also reference modules (notebook 05): `depends_on = [module.network]` means "wait for everything in the network module."


## The `lifecycle` meta-argument

Every resource accepts a `lifecycle { ... }` block that controls *how* Terraform replaces, destroys, or ignores changes to it. Four arguments inside, each solving a specific class of problem.

```hcl
resource "aws_instance" "web" {
  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.micro"

  lifecycle {
    create_before_destroy = true
    prevent_destroy       = false
    ignore_changes        = [tags["LastScanned"]]
    replace_triggered_by  = [aws_security_group.web.id]
  }
}
```

We walk each one.


### `create_before_destroy`

When Terraform replaces a resource (because an immutable attribute changed), the default order is **destroy first, then create**. This is the safe order for most resources — destroying a network interface frees the IP before the new one tries to claim it.

For resources that hold state or that must remain reachable, this is the wrong order. Replacing an EC2 instance with destroy-first means the old one is gone for the seconds-to-minutes the new one takes to come up. **`create_before_destroy = true`** flips the order: the new resource is created and healthy before the old one is destroyed.

```hcl
resource "aws_launch_template" "web" {
  name_prefix   = "web-"
  image_id      = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.micro"

  lifecycle {
    create_before_destroy = true
  }
}
```

**When it works.** Resources whose names you can let Terraform generate (use `name_prefix`, not `name`) so the new resource doesn't collide with the old. Resources without persistent storage tied to a single ID. Autoscaling launch templates, load balancer target groups, EC2 instances behind an ASG.

**When it doesn't.** Anything with a unique global name you can't randomize (S3 buckets, some database identifiers). The old and new can't coexist because they'd both want the same name. The fix is *renaming the resource entirely* between deploys, or accepting brief downtime.


### `prevent_destroy`

A safety belt for resources that must not be accidentally destroyed.

```hcl
resource "aws_db_instance" "prod" {
  identifier = "prod-db"
  # ... a hundred other settings ...

  lifecycle {
    prevent_destroy = true
  }
}
```

If any operation (a `terraform destroy`, or a config change that would force replacement) would destroy this resource, Terraform errors out at plan time:

```
Error: Instance cannot be destroyed

Resource aws_db_instance.prod has lifecycle.prevent_destroy
set, but the plan calls for this resource to be destroyed.
To avoid this error and continue with the plan, either disable
lifecycle.prevent_destroy or reduce the scope of the plan...
```

This is the right default for production-critical state — databases, S3 buckets holding customer data, KMS keys, anything where "oops" means restoring from backup or recreating from scratch. It costs you nothing; it stops you from one of the worst possible Terraform mistakes.

**Caveat.** `prevent_destroy` blocks *Terraform-initiated* destruction. It does not prevent destruction from outside Terraform (the AWS console, `aws rds delete-db-instance`). That's a separate problem solved with IAM and deletion protection on the resource itself.


### `ignore_changes`

Some attributes get modified by something outside Terraform — autoscaling adjusts a tag, a CI pipeline updates an AMI, an external tool adds an annotation. Terraform sees the difference between desired and actual state and tries to "fix" it by reverting the change. You don't want that.

**`ignore_changes`** tells Terraform to leave specific attributes alone:

```hcl
resource "aws_instance" "web" {
  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.micro"

  tags = {
    Name = "web"
  }

  lifecycle {
    ignore_changes = [
      ami,                       # we update AMI out-of-band via blue-green
      tags["LastScanned"],       # security scanner writes this tag
    ]
  }
}
```

The list takes attribute paths. Bare names for top-level attributes (`ami`). Bracket-accessor syntax for collection elements (`tags["LastScanned"]`). `[all]` to ignore *every* attribute (rarely the right answer — at that point the resource isn't really managed by Terraform).

**The trap.** `ignore_changes` doesn't just suppress the plan diff — it tells Terraform to *not update* that attribute even when the HCL value changes. Edit the AMI in your config, and Terraform won't update the instance. Use `ignore_changes` only for attributes you genuinely *don't* want Terraform to control going forward.


### `replace_triggered_by` (Terraform 1.2+)

Sometimes you want a resource to be *replaced* when an unrelated value changes, even if no attribute reference forces it. The classic case: redeploy a Lambda when its source-code hash changes, even though the hash isn't an argument the Lambda block sees directly.

```hcl
resource "aws_lambda_function" "worker" {
  function_name    = "worker"
  filename         = "worker.zip"
  source_code_hash = filebase64sha256("worker.zip")
  role             = aws_iam_role.lambda.arn
  handler          = "index.handler"
  runtime          = "python3.11"

  lifecycle {
    replace_triggered_by = [terraform_data.worker_build.id]
  }
}

resource "terraform_data" "worker_build" {
  input = filebase64sha256("worker.zip")
}
```

When the zip changes, `terraform_data.worker_build`'s `id` changes (it's derived from `input`), which triggers replacement of the Lambda. This is the modern idiom for "rebuild when source changes," replacing older hacks that abused `null_resource` triggers.

`terraform_data` is a no-op resource (Terraform 1.4+) whose only purpose is to carry an `input` value and produce an `id` that depends on it. It's the right tool for this kind of derivation.


## `count` — basic iteration

Sometimes you want N copies of a resource. The oldest answer in Terraform is **`count`**:

```hcl
resource "aws_instance" "web" {
  count = 3

  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.micro"

  tags = {
    Name = "web-${count.index}"
  }
}
```

This creates `aws_instance.web[0]`, `aws_instance.web[1]`, `aws_instance.web[2]`. Inside the block, `count.index` is the zero-based index of the current instance.

**Where `count` works.** Identical resources that differ only by index (`web-0`, `web-1`, `web-2`). Cases where the count itself is the variable (`count = var.instance_count`).

**Where `count` breaks.** The day you change `count` from 3 to 2 by removing `web-1` from a list of names, Terraform doesn't know you wanted to remove the *middle* one. It sees a list of length 2 and assumes you wanted indices 0 and 1 — so it *renames* `web-2` to `web-1` (replacing in-place) and destroys the old `web-1`. This is the classic "shifted index" disaster: you intended to remove one instance, you actually destroyed and recreated two.

The fix is **`for_each`**.


## `for_each` — the better iteration

**`for_each`** keys resources by a string or by a set member rather than by integer index. The result is stable across reorderings.

```hcl
resource "aws_instance" "web" {
  for_each = toset(["alpha", "beta", "gamma"])

  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.micro"

  tags = {
    Name = "web-${each.key}"
  }
}
```

This creates `aws_instance.web["alpha"]`, `aws_instance.web["beta"]`, `aws_instance.web["gamma"]`. Inside the block, `each.key` is the map key (or set member); `each.value` is the value.

Remove `"beta"` from the set and `aws_instance.web["beta"]` is destroyed. `web["alpha"]` and `web["gamma"]` are *untouched*. No shifting, no surprise replacement.

`for_each` also accepts a map, which is useful when each instance needs different attributes:

```hcl
resource "aws_subnet" "main" {
  for_each = {
    public_a  = { cidr = "10.0.1.0/24", az = "us-east-1a" }
    public_b  = { cidr = "10.0.2.0/24", az = "us-east-1b" }
    private_a = { cidr = "10.0.3.0/24", az = "us-east-1a" }
  }

  vpc_id            = aws_vpc.main.id
  cidr_block        = each.value.cidr
  availability_zone = each.value.az
  tags = {
    Name = each.key
  }
}
```

Three subnets, each named, each with its own CIDR and AZ. Add a fourth subnet to the map and only that one is created. Rename a key and Terraform sees a destroy-and-create — which is correct because keys are identity.

**`for_each` requires the keys to be known at plan time.** A `for_each` over a value that depends on a resource that hasn't been created yet is an error: "The for_each value depends on resource attributes that cannot be determined until apply." The fix is usually to compute the keys from inputs (variables, locals) rather than from other resources.


## `count` vs `for_each` — the decision

| Use case | Reach for |
|---|---|
| You want N identical resources and N might change | **`for_each` over a set of stable keys** |
| You want a configurable number of resources by index | `count` (rarely a good fit in practice) |
| Conditional creation — a resource that exists when a flag is true | `count = var.enabled ? 1 : 0` is the idiomatic toggle |
| One resource per item in a list, where order matters | `count` over the list, indexing by `count.index` |
| One resource per named thing, where reordering should be safe | **`for_each`** over a map or set keyed by the name |

The honest default: **prefer `for_each`** unless you have a specific reason. The "I removed an item and it destroyed half my fleet" disaster only happens with `count`.

The exception is the conditional pattern. `count = var.enabled ? 1 : 0` is shorter than the `for_each` equivalent (`for_each = var.enabled ? toset(["on"]) : toset([])`), and you reference the resource as `aws_instance.web[0]` which is acceptable for a singleton.


## The `-/+` replace, revisited

Most of this notebook is *about* preventing the surprise `-/+` in `terraform plan`. The notebook 01 cheat sheet bears repeating:

| Plan symbol | Meaning |
|---|---|
| `+` create | Will be created |
| `~` update in-place | Attribute changed; resource stays |
| `-` destroy | Will be destroyed |
| `-/+` replace | Will be destroyed and recreated |

The `-/+` is the dangerous one. Producing it accidentally is how teams end up with five-minute outages, lost data, or replaced load balancers mid-traffic. Three habits prevent most of them:

- **Read every plan.** Especially in production. CI gates should refuse to apply a plan with `-/+` on certain resource types without an explicit approval flag.
- **Know which arguments are "ForceNew" before you change them.** The provider docs list them. Database engine version, VPC CIDR, instance type on some platforms, KMS key policies. If a deployment plan changes one of these, expect replacement.
- **Use `create_before_destroy` on replaceable-but-stateful resources.** Launch templates, target groups, instances behind autoscaling. The new one is healthy before the old one dies.

When the plan shows a `-/+` you didn't expect, the answer is *not* to apply it and find out. Investigate. The Terraform community has produced more outage post-mortems for "I should have read the plan more carefully" than for any other reason.


## A worked example — a VPC with three subnets

Putting it all together: one VPC, three subnets driven by `for_each`, an internet gateway, and a route table. Real, idiomatic Terraform 1.6+.

```hcl
terraform {
  required_version = ">= 1.6"
  required_providers {
    aws = { source = "hashicorp/aws", version = "~> 5.0" }
  }
}

provider "aws" {
  region = "us-east-1"
}

locals {
  subnets = {
    public_a = { cidr = "10.0.1.0/24", az = "us-east-1a" }
    public_b = { cidr = "10.0.2.0/24", az = "us-east-1b" }
    public_c = { cidr = "10.0.3.0/24", az = "us-east-1c" }
  }
}

resource "aws_vpc" "main" {
  cidr_block           = "10.0.0.0/16"
  enable_dns_hostnames = true
  tags = { Name = "main" }

  lifecycle {
    prevent_destroy = true   # safety belt — VPC change is a big deal
  }
}

resource "aws_internet_gateway" "main" {
  vpc_id = aws_vpc.main.id
  tags   = { Name = "main" }
}

resource "aws_subnet" "public" {
  for_each = local.subnets

  vpc_id                  = aws_vpc.main.id
  cidr_block              = each.value.cidr
  availability_zone       = each.value.az
  map_public_ip_on_launch = true
  tags                    = { Name = each.key }
}

resource "aws_route_table" "public" {
  vpc_id = aws_vpc.main.id

  route {
    cidr_block = "0.0.0.0/0"
    gateway_id = aws_internet_gateway.main.id
  }

  tags = { Name = "public" }
}

resource "aws_route_table_association" "public" {
  for_each = aws_subnet.public

  subnet_id      = each.value.id
  route_table_id = aws_route_table.public.id
}
```

Walk through what's happening.

- `aws_vpc.main` is the root of the dependency graph. Everything else references it. `prevent_destroy = true` means a future `terraform destroy` errors out before touching it.
- `aws_subnet.public` is created with `for_each` over the `local.subnets` map. One subnet per entry, keyed by name. Adding `public_d` to the map adds one subnet; removing `public_b` removes one subnet, *without* shifting the others.
- `aws_route_table_association.public` does the same `for_each`, this time keyed by `aws_subnet.public` itself. `each.value.id` is each subnet's ID. The associations parallel the subnets one-to-one.
- The dependency graph: VPC → IGW → route table → associations, with subnets attached to the VPC in parallel and attached to the route table after both exist. Terraform builds this graph automatically from the references.

The plan reports 9 resources to create (1 VPC + 1 IGW + 3 subnets + 1 route table + 3 associations). Apply runs the independents in parallel — subnets get created concurrently with the route table, then the associations once both are ready.

Add `public_d = { cidr = "10.0.4.0/24", az = "us-east-1d" }` to `local.subnets` and the plan reports 2 to add: one subnet, one association. No other resources are touched.


## Forcing replacement — `terraform apply -replace`

Occasionally you want to *force* Terraform to replace a resource even when the config hasn't changed — the instance has gotten weird, the database has a stuck connection, an EC2 instance type needs to be cycled. The modern command is **`terraform apply -replace=<address>`**:

```bash
$ terraform apply -replace='aws_instance.web["alpha"]'
```

This plans `-/+` replacement of that one resource, regardless of whether its config would otherwise change. It's the safer modern replacement for the older **`terraform taint`** command, which mutated state directly and gave no preview.

`-replace` is plan-and-apply: you see what will happen before confirming. Use it for cycling specific instances, not for bulk operations.

`terraform taint` and `untaint` still exist but are deprecated. Don't reach for them in new work.


## Forward

Notebook three turns to **State Management** — what the state file is, why local state hurts in teams, the standard remote backends (S3 + DynamoDB, Terraform Cloud), state locking and what split-brain looks like, the drift problem and how to detect it, sensitive values and how they leak through state, and the state-surgery commands (`state mv`, `state rm`, `import`) for when the configuration and the cloud disagree and you have to fix it by hand.
